In [ ]:
# --- Experiment Control Center ---
# This notebook is the single interface for managing all experiment workflows:
# 1. Generate Profiles: Choose between a full Grid Search or an intelligent HPO sweep.
# 2. Run Experiments: Execute a chosen profile using the main pipeline.
# 3. Analyze & Report: Collate results, report back to Optuna, and visualize findings.

# --- Imports ---
import sys
import os
import pandas as pd
from pathlib import Path

# Setup paths to import our modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import our custom utilities
from hpo.profile_generators import generate_grid_search_profile
from hpo.optuna_utils import generate_trials_from_study, report_results_to_study
from hpo.search_space import define_search_space, define_fast_search_space

import optuna.visualization as vis

# --- Display Settings ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

print("✅ Setup complete. Experiment Control Center is ready.")

In [ ]:
# --- CONTROL PANEL for HPO ---
# Set to 'FAST' for quick debugging runs or 'FULL' for production HPO.
HPO_MODE = 'FAST' 

# Select which function to use based on the mode
if HPO_MODE == 'FAST':
    search_space_to_use = define_fast_search_space
    study_name_suffix = "fast_debug"
    trials_to_generate = 5
else: # 'FULL'
    search_space_to_use = define_search_space
    study_name_suffix = "full_sweep"
    trials_to_generate = 30

# Define the HPO study parameters
STUDY_NAME = f"main_pinn_{study_name_suffix}"
FIXED_PARAMS = {"seed": 42} # 'data_sample' is now part of the search space

# Define the output file path
hpo_output_file = f'profiles/{STUDY_NAME}_batch_{trials_to_generate}.csv'
# --- Generate the trials using the selected search space function ---
generated_trials_df = generate_trials_from_study(
    study_name=STUDY_NAME,
    n_trials=trials_to_generate,
    output_file=hpo_output_file,
    search_space_func=search_space_to_use, # <-- Pass the selected function here
    **FIXED_PARAMS
)

print("\nGenerated HPO trial configurations:")
display(generated_trials_df.head())

# Set this as the profile to run in the next step
PROFILE_TO_RUN = hpo_output_file